<div style="background-color: #FFFF00; color: #000000; padding: 12px 18px; font-family: Arial, Helvetica, sans-serif;">
<strong style="color: #000000;">MINE-4101 Ciencia de Datos Aplicada · Universidad de los Andes · 2026-20</strong><br>
<span style="color: #000000;">Proyecto final · Primera entrega · Actividad 6</span>
</div>

# Entendimiento de los datos

Análisis exploratorio y calidad de los datos del levantamiento catastral de un municipio colombiano.

**Integrantes**

- Daniel Felipe Triviño Santana
- Nicolas Bedoya Figueroa
- Joubert Daniel Álvarez
- Gabriel Bejarano

**Datos:** la base catastral de un municipio colombiano, en el modelo LADM-COL y en su versión podada: 45 tablas, de las cuales 20 tienen los datos del levantamiento (410 171 filas) y 23 son dominios. Ver la [descripción de la base](../../datos/base-podada/README.md) y su [diccionario de datos](../../datos/base-podada/diccionario-de-datos.md).

**Qué pide el enunciado** (actividad 6, 35 % de la primera entrega): un reporte de análisis exploratorio y de calidad de los datos, en el que sea evidente el uso de técnicas univariadas y multivariadas, gráficas y no gráficas. El objetivo de la entrega es validar la calidad y la suficiencia de los datos para la solución planteada.

**Estado:** en desarrollo. Están la estructura, la carga de librerías y la de los datos (etapa 2); las etapas 3 a 8 están por desarrollar. Los resultados más importantes pasan a la sección 6 del [reporte](../reporte/reporte.pdf).

Última actualización: octubre de 2026.

## Tabla de contenido

Las etapas siguen el proceso de exploración que presenta el curso (Ghosh et al., 2018): cargar los datos, distinguir los atributos, analizarlos uno por uno, buscar interacciones entre ellos, detectar valores aberrantes y faltantes, detectar valores atípicos y recoger los insights. El paso de ingeniería de características de ese proceso corresponde a la preparación de datos de la entrega final (actividad 8) y no se desarrolla aquí.

Los cuatro tipos de análisis que pide el enunciado (univariado y multivariado, gráfico y no gráfico) son las subsecciones de las etapas 4 y 5.

| | Etapa | Qué se hace | Apoyo en el curso |
| --- | --- | --- | --- |
| 1 | [Librerías y configuración](#librerias) | Cargar las librerías y fijar las opciones de tablas y gráficas | Tutoriales 1 y 2 |
| 2 | [Carga de los datos](#carga) | Leer las tablas y revisar dimensiones, tipos y una muestra | Inicio del proceso de exploración; tutoriales 1 y 2 |
| 3 | [Identificación de los atributos](#atributos) | Clasificar cada atributo como cuantitativo o cualitativo | Paso 1; EDA, diapositivas 10 a 16 |
| 4 | [Análisis univariado](#univariado) | Describir cada atributo por separado: 4.1 no gráfico, 4.2 gráfico | Paso 2; EDA, diapositivas 17 a 28 y 57 a 60; tutorial 1 |
| 5 | [Análisis multivariado](#multivariado) | Buscar relaciones entre atributos: 5.1 no gráfico, 5.2 gráfico | Paso 3; EDA, diapositivas 29 a 47 y 61 a 70; tutorial 1 |
| 6 | [Calidad de los datos](#calidad) | 6.1 valores faltantes y aberrantes, 6.2 valores atípicos, 6.3 resumen de problemas | Pasos 4 y 5; Calidad, diapositivas 4 a 22; tutorial 2 |
| 7 | [Suficiencia de los datos](#suficiencia) | Concluir si los datos alcanzan para la solución planteada | Objetivos de la primera entrega, en el enunciado |
| 8 | [Conclusiones e insights](#conclusiones) | Reunir los hallazgos de las etapas anteriores | Cierre del proceso de exploración; EDA, diapositivas 4 a 6 |

Fuentes del curso:

- **EDA:** diapositivas «Análisis exploratorio de datos (EDA)», clases 3 y 4.
- **Calidad:** diapositivas «Calidad y limpieza de datos», clase 5.
- **Tutoriales 1 y 2:** [data-analysis-tutorial](https://github.com/datascience-uniandes/data-analysis-tutorial) y [data-quality-tutorial](https://github.com/datascience-uniandes/data-quality-tutorial).
- **Proceso de exploración:** A. Ghosh et al., "A comprehensive review of tools for exploratory analysis of tabular industrial datasets", *Visual Informatics* 2(4), 2018. Es el que muestra la diapositiva 9 de EDA.

<a id="librerias"></a>

## 1. Librerías y configuración

Las librerías son las que usan los tutoriales del curso y están en el entorno del proyecto (ver el [README](../../README.md#entorno-de-python)). Las que solo necesita una etapa (`scipy`, `statsmodels`, `scikit-learn`, `ydata-profiling`, `Levenshtein`) se agregan a esta primera celda cuando esa etapa se desarrolle.

In [1]:
# Librerías básicas de análisis y visualización
import platform
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import seaborn as sns
from matplotlib import font_manager

In [2]:
# Raíz del repositorio: la carpeta que contiene pyproject.toml
RAIZ = next(c for c in [Path.cwd(), *Path.cwd().parents] if (c / "pyproject.toml").exists())

# Tablas: todas las columnas, hasta 100 filas y sin notación científica
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.options.display.float_format = "{:.2f}".format

# Gráficas: identidad visual del proyecto (ver identidad/README.md)
plt.style.use(RAIZ / "identidad" / "graficas.mplstyle")
AMARILLO, NEGRO, GRIS = "#FFFF00", "#000000", "#666666"

# Semilla para todo lo que use azar (muestras, reducción de dimensionalidad)
SEMILLA = 4101

In [3]:
# Versiones con las que se ejecutó el notebook
print(f"Python {platform.python_version()}")
for libreria in (np, pd, pa, mpl, sns):
    print(f"{libreria.__name__} {libreria.__version__}")

# Fuente que usan las gráficas: debería ser Arial (ver identidad/README.md)
ruta_fuente = font_manager.findfont(font_manager.FontProperties(family=plt.rcParams["font.family"]))
print("Fuente de las gráficas:", font_manager.get_font(ruta_fuente).family_name)

Python 3.12.13
numpy 2.3.5
pandas 2.3.3
pyarrow 26.0.0
matplotlib 3.10.0
seaborn 0.13.2
Fuente de las gráficas: Arial


<a id="carga"></a>

## 2. Carga de los datos

Como en los tutoriales del curso: leer cada tabla y revisar sus dimensiones, los tipos de sus columnas y una muestra de filas.

**Qué se lee.** El archivo `datos/predios/predios.parquet`: la base podada en un solo archivo, con un predio por fila y sin datos personales. Lo que la base guarda sobre los interesados y sobre quienes atendieron la visita llega reducido a conteos por predio. Cómo se construye el archivo y qué significa cada columna está en [datos/predios/](../../datos/predios/README.md).

**El archivo no está en el repositorio.** Sigue identificando al municipio, así que el equipo lo comparte por un canal privado. Para ejecutar el notebook hay que pedirlo o construirlo a partir de la base podada, y dejarlo en `datos/predios/`. Sin él, la primera celda de esta etapa se detiene con un mensaje que lo explica.

**Tres tablas.** Lo que en el archivo está anidado se separa aquí en tablas que se unen por `id_predio`:

| Tabla | Una fila por | De dónde sale |
| --- | --- | --- |
| `predios` | Predio | Las columnas simples del archivo y los campos de sus estructuras (`terreno`, `ph_matriz`, `titulares`), con el nombre de la estructura como prefijo: `terreno.area_terreno` |
| `construcciones` | Construcción | La lista `construcciones` de cada predio |
| `unidades` | Unidad de construcción | La lista `unidades` de cada predio, con las características de cada unidad |

En propiedad horizontal, la construcción de una unidad suele estar en el predio matriz: `unidades.id_construccion` se une con `construcciones.id_construccion`, no necesariamente dentro del mismo predio.

**Lo que las salidas no pueden mostrar.** El repositorio es público y las salidas de las celdas se suben con el notebook (ver [Antes de compartirla](../../datos/base-podada/README.md#antes-de-compartirla)). Ninguna debe dejar a la vista:

- Lo que identifica al municipio: los códigos `departamento` y `municipio`, el `numero_predial` (sus primeros dígitos son esos códigos), el círculo registral (`codigo_orip`), la matrícula, el código homologado, el `id_operacion` y el `ente_emisor`, que nombra notarías y juzgados del lugar. Tampoco mapas: el archivo no trae coordenadas.
- Los textos libres (`nombre`, `observaciones`, `numero_fuente`, `observacion_fuente` y las observaciones de las unidades), que pueden traer nombres de personas.

Esas columnas están en `COLUMNAS_RESERVADAS`. Se pueden usar en los cálculos, pero no se muestran sus valores.

In [4]:
# El archivo se construye con datos/predios/ y no está en el repositorio
ARCHIVO_PREDIOS = RAIZ / "datos" / "predios" / "predios.parquet"
if not ARCHIVO_PREDIOS.exists():
    raise FileNotFoundError(
        f"Falta {ARCHIVO_PREDIOS.relative_to(RAIZ)}: pídalo al equipo o constrúyalo "
        "como indica datos/predios/README.md"
    )

# flatten() convierte cada estructura en columnas con su prefijo (terreno.area_terreno);
# las listas (construcciones, unidades) quedan como columnas de listas
predios = pq.read_table(ARCHIVO_PREDIOS).flatten().to_pandas()

# Las fechas llegan como objetos date de Python
for columna in ["fecha_visita_predial", "fecha_inicio_tenencia", "fecha_documento_fuente"]:
    predios[columna] = pd.to_datetime(predios[columna])

# Columnas que identifican al municipio o pueden traer nombres: no se muestran sus valores
COLUMNAS_RESERVADAS = [
    "departamento", "municipio", "numero_predial", "codigo_orip", "matricula_inmobiliaria",
    "codigo_homologado", "id_operacion", "ente_emisor",
    "nombre", "observaciones", "numero_fuente", "observacion_fuente",
]

In [5]:
def desanidar(columna):
    """Una fila por elemento de una columna de listas, con el id del predio al que pertenece."""
    filas = predios[["id_predio", columna]].explode(columna).dropna(subset=[columna])
    return pd.concat(
        [filas[["id_predio"]].reset_index(drop=True), pd.json_normalize(filas[columna].tolist())],
        axis=1,
    )


construcciones = desanidar("construcciones")
unidades = desanidar("unidades")
predios = predios.drop(columns=["construcciones", "unidades"])

TABLAS = {"predios": predios, "construcciones": construcciones, "unidades": unidades}

In [6]:
# Dimensiones de cada tabla
pd.DataFrame(
    {"filas": [len(t) for t in TABLAS.values()], "columnas": [t.shape[1] for t in TABLAS.values()]},
    index=list(TABLAS),
)

,filas,columnas
predios,22220,73
construcciones,13483,15
unidades,42988,21


In [7]:
# Tipos de las columnas y cuántos valores no nulos tiene cada una
for nombre, tabla in TABLAS.items():
    print(f"=== {nombre}")
    tabla.info()
    print()

=== predios
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22220 entries, 0 to 22219
Data columns (total 73 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   id_predio                                22220 non-null  int64         
 1   departamento                             22220 non-null  object        
 2   municipio                                22220 non-null  object        
 3   numero_predial                           22220 non-null  object        
 4   zona                                     22220 non-null  object        
 5   sector                                   22220 non-null  object        
 6   comuna                                   22220 non-null  object        
 7   barrio                                   22220 non-null  object        
 8   manzana_vereda                           22220 non-null  object        
 9   numero_terreno             

In [8]:
# Muestra de predios, sin las columnas reservadas
predios.drop(columns=COLUMNAS_RESERVADAS).sample(5, random_state=SEMILLA)

,id_predio,zona,sector,comuna,barrio,manzana_vereda,numero_terreno,condicion_propiedad,edificio,piso,unidad,avaluo_catastral,tipo,condicion_predio,destinacion_economica,area_registral_m2,fecha_visita_predial,resultado_visita,tipo_novedad_numero_predial,n_contactos_visita,autoriza_notificaciones,tipo_direccion,es_direccion_principal,ph_matriz.area_total_terreno,ph_matriz.area_total_terreno_privada,ph_matriz.area_total_terreno_comun,ph_matriz.area_total_construida,ph_matriz.area_total_construida_privada,ph_matriz.area_total_construida_comun,ph_matriz.numero_torres,ph_matriz.total_unidades_privadas,id_predio_matriz,coeficiente_copropiedad,n_unidades_prediales,ids_predios_informales,ids_predios_formales,terreno.id_terreno,terreno.area_terreno,terreno.area_geometrica,terreno.perimetro,terreno.compacidad,terreno.n_vertices,terreno.n_poligonos,terreno.geometria_valida,tipo_derecho,fraccion_derecho,fecha_inicio_tenencia,tipo_fuente,estado_disponibilidad,tipo_principal,fecha_documento_fuente,titulares.es_agrupacion,titulares.tipo_agrupacion,titulares.n_titulares,titulares.n_persona_natural,titulares.n_persona_juridica,titulares.n_femenino,titulares.n_masculino,titulares.n_sexo_sin_determinar,titulares.participacion_max,titulares.max_predios_mismo_titular
3746,161482,00,00,00,00,0005,0615,0,00,00,0000,24149000.00,Predio.Privado,NPH,Agricola,75.99,2025-09-09,Exitoso,None,1,False,No_Estructurada,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,[],[],83472.00,79.20,79.18,47.05,0.45,6.00,1.00,True,Dominio,1.00,2015-09-29,Documento_Publico.Sentencia_Judicial,Disponible,Documento,2015-06-24,True,Grupo_Civil,3,3,0,0,3,0,0.33,2
20415,467099,00,00,00,00,0002,1098,9,02,01,0015,233893000.00,Predio.Privado,PH.Unidad_Predial,Habitacional,98.38,2025-12-23,Incompleto,Desenglobe,0,None,No_Estructurada,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,162190.00,0.01,0,[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Dominio,1.00,2022-09-19,Documento_Publico.Escritura_Publica,Disponible,Documento,2022-08-25,True,Grupo_Civil,2,2,0,1,1,0,0.50,1
10522,437420,00,00,00,00,0004,3451,7,00,00,1062,0.00,Predio.Privado,Parque_Cementerio.Unidad_Predial,Servicios_Funerarios,3.00,2026-01-07,Incompleto,Predio_Nuevo,0,None,No_Estructurada,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,161773.00,0.00,0,[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Dominio,1.00,2016-08-17,Documento_Publico.Escritura_Publica,Disponible,Documento,2016-07-29,True,Grupo_Civil,4,4,0,3,1,0,0.25,1
11313,439793,00,00,00,00,0004,3665,9,08,01,0008,614183000.00,Predio.Privado,PH.Unidad_Predial,Comercial,1242.42,2025-07-18,Incompleto,Cambio_Numero_Predial,0,None,No_Estructurada,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164518.00,0.01,0,[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Dominio,1.00,2012-10-17,Documento_Publico.Escritura_Publica,Disponible,Documento,2012-08-01,False,None,1,0,1,0,0,1,1.00,78
2579,159142,00,00,00,00,0006,0541,0,00,00,0000,215751000.00,Predio.Privado,NPH,Agricola,0.00,2025-09-03,Sin_Visita,None,1,False,No_Estructurada,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,[],[],82180.00,679.00,679.00,287.97,0.10,37.00,1.00,True,Dominio,1.00,2000-12-26,Documento_Publico.Escritura_Publica,Disponible,Documento,2000-12-14,False,None,1,1,0,0,1,0,1.00,1


In [9]:
# Muestra de construcciones
construcciones.sample(5, random_state=SEMILLA)

,id_predio,id_construccion,identificador,total_pisos,total_sotanos,total_mezanines,total_semisotanos,area_total_construccion,altura_total_construccion,area_geometrica,perimetro,compacidad,n_vertices,n_poligonos,geometria_valida
12693,470348,52513,A,1,0,0,0,164.10,3.00,164.11,87.30,0.27,39,1,True
8611,165576,55281,A,2,0,0,0,6359.10,6.00,6359.11,1893.09,0.02,681,26,True
106,155783,50533,D,2,0,0,0,188.70,6.00,188.71,56.52,0.74,8,1,True
7338,164006,52980,C,1,0,0,0,137.50,3.00,137.46,47.89,0.75,5,1,True
49,155711,57474,A,1,0,0,0,45.60,3.00,45.64,31.58,0.58,5,1,True


In [10]:
# Muestra de unidades de construcción, sin sus observaciones
unidades.drop(columns=["observaciones"]).sample(5, random_state=SEMILLA)

,id_predio,id_unidad,id_construccion,identificador,tipo_unidad_construccion,uso,tipo_planta,planta_ubicacion,total_plantas,anio_construccion,area_construida,area_construida_caracteristicas,area_privada_construida,altura,area_geometrica,perimetro,compacidad,n_vertices,n_poligonos,geometria_valida
34832,458552,123726,60837,A,Anexo,Anexo.Marquesinas_Patios_Cubiertos,Piso,3,1,2023,7.20,0.00,7.20,3,7.22,12.55,0.58,7,1,True
36911,463070,110018,54250,C,Residencial,Residencial.Vivienda_Hasta_3_Pisos_En_PH,Piso,1,1,2024,34.80,0.00,34.80,3,34.77,24.68,0.72,7,1,True
28866,448019,107706,63109,F,Residencial,Residencial.Vivienda_Hasta_3_Pisos_En_PH,Piso,2,1,2019,62.20,0.00,62.20,3,62.16,34.80,0.65,13,1,True
6862,160497,116167,55254,D,Industrial,Industrial.Industrias,Piso,1,1,2000,71.40,71.40,NaN,3,71.42,35.27,0.72,5,1,True
19730,172700,127207,56482,A,Residencial,Residencial.Vivienda_Hasta_3_Pisos_En_PH,Piso,3,3,2022,23.60,0.00,23.60,3,23.58,25.14,0.47,9,1,True


<a id="atributos"></a>

## 3. Identificación de los atributos

Primer paso del proceso de exploración: distinguir de qué tipo es cada atributo, porque de eso depende qué medidas y qué gráficas le sirven.

| Tipo | Subtipos | Cómo reconocerlo |
| --- | --- | --- |
| Cuantitativo | Discreto o continuo | Representa una magnitud: restar dos valores tiene sentido. Es continuo si además tiene sentido el valor medio entre dos valores |
| Cualitativo (categórico) | Nominal u ordinal | Toma niveles. Es ordinal si los niveles se pueden ordenar |

En esta base, el tipo de dato de la columna no basta para clasificarla:

- Los atributos de dominio (`condicion_predio`, `uso`, `tipo`) guardan el `t_id` de una fila de la tabla del dominio. Son cualitativos, aunque en la base sean números.
- Las llaves y los identificadores (`t_id`, `numero_predial`, `matricula_inmobiliaria`) tampoco son cuantitativos: restar dos de ellos no significa nada.

In [11]:
# Pendiente

<a id="univariado"></a>

## 4. Análisis univariado

Describe cada atributo por separado.

El tutorial 1 incluye un perfilamiento automático (`ydata-profiling`), que sirve como primera mirada a todas las columnas de una tabla. Su reporte en HTML incluye muestras de los datos: no se sube al repositorio.

### 4.1 No gráfico

Medidas estadísticas que resumen la distribución de un atributo:

- **Tabla de frecuencias:** valores únicos, valor más común (moda), frecuencia absoluta y relativa.
- **Tendencia central:** media y mediana. La mediana es robusta a los valores extremos.
- **Dispersión:** varianza y desviación estándar; la desviación absoluta mediana (MAD) como alternativa robusta.
- **Posición:** cuantiles, rango intercuartílico, mínimo y máximo.
- **Forma:** asimetría y curtosis.

In [12]:
# Pendiente

### 4.2 Gráfico

- **Barras** para las frecuencias de un atributo cualitativo.
- **Histograma** para la distribución de uno cuantitativo. Hay que elegir el número de intervalos.
- **Diagrama de caja** para la tendencia central, la asimetría y los valores atípicos. No muestra bien las distribuciones con varias modas.

In [13]:
# Pendiente

<a id="multivariado"></a>

## 5. Análisis multivariado

Describe la distribución conjunta de varios atributos o la relación entre ellos. En la práctica casi siempre se hace de dos en dos.

### 5.1 No gráfico

- **Tablas cruzadas** entre atributos cualitativos, o cuantitativos con pocos valores.
- **Correlación** entre atributos cuantitativos: lineal (Pearson) y monótona (Spearman), y la matriz de correlación cuando son varios. Correlación no implica causalidad.
- **Regresión lineal**, para estimar un atributo a partir de otros y leer sus coeficientes. Tampoco implica causalidad y tiene supuestos que deben cumplirse.
- **Análisis de componentes principales (PCA)**, para resumir muchos atributos cuantitativos en pocos componentes.
- **Pruebas de hipótesis** (t, ANOVA, chi-cuadrado; clase 6 y tutorial 3), cuando haga falta confirmar una diferencia o una asociación.

In [14]:
# Pendiente

### 5.2 Gráfico

- **Diagrama de dispersión** entre dos atributos cuantitativos, con el color o la forma de los puntos para una categoría.
- **Mapa de calor** para una magnitud según dos atributos cualitativos, o para la matriz de correlación.
- **Gráficas por categoría:** la distribución de un atributo cuantitativo en cada nivel de uno cualitativo.
- **Reducción de dimensionalidad** para ver muchos atributos en dos dimensiones. PCA y t-SNE están en `scikit-learn`; UMAP necesitaría una dependencia nueva.
- **Mapas**, solo si la posición geográfica es relevante. Aquí, además, un mapa del municipio lo identifica (ver la [etapa 2](#carga)).

In [15]:
# Pendiente

<a id="calidad"></a>

## 6. Calidad de los datos

El curso define nueve dimensiones de calidad: completitud, consistencia, conformidad, relevancia, precisión, procesabilidad, disponibilidad, temporalidad y credibilidad. Y clasifica los problemas según el nivel en el que aparecen:

| Nivel | Problemas |
| --- | --- |
| Atributo | Valor faltante, violación de sintaxis, error de digitación, violación de dominio |
| Registro | Violación de integridad: valores de un mismo registro que se contradicen |
| Columna | Sinónimos, homónimos, violación de singularidad |
| Tabla | Duplicados aproximados |
| Múltiples tablas | Representación heterogénea de los datos, sinónimos, granularidades distintas |

Aquí los problemas se diagnostican. La limpieza corresponde a la preparación de datos de la entrega final (actividad 8).

### 6.1 Valores faltantes y aberrantes

Paso 4 del proceso de exploración. El [diccionario de datos](../../datos/base-podada/diccionario-de-datos.md) ya trae, para cada columna, el porcentaje de filas que tienen dato; es el punto de partida para la completitud. Técnicas del tutorial 2 para lo demás:

- Diccionarios de valores válidos. Aquí ese papel lo cumplen las tablas de dominio.
- Expresiones regulares para comprobar formatos.
- Duplicados, por un conjunto de atributos o por el registro entero.
- Medidas de similitud entre textos (distancia de Levenshtein) para los errores de digitación.
- Consistencia entre columnas del mismo registro.

In [16]:
# Pendiente

### 6.2 Valores atípicos

Paso 5 del proceso de exploración. Las dos definiciones estándar del curso: valores a más de 3 desviaciones estándar de la media, o a más de 1,5 veces el rango intercuartílico por debajo del primer cuartil o por encima del tercero. Qué cuenta como atípico depende muchas veces del negocio.

In [17]:
# Pendiente

### 6.3 Resumen de problemas de calidad

Como pide la conclusión del tutorial 2: cada problema encontrado, la dimensión de calidad que afecta y la estrategia para resolverlo o mitigarlo.

| Problema | Tabla y atributo | Dimensión | Nivel | Estrategia |
| --- | --- | --- | --- | --- |
| Pendiente | | | | |

<a id="suficiencia"></a>

## 7. Suficiencia de los datos

Responde, con lo encontrado en las etapas anteriores, si los datos alcanzan para la solución planteada: si están los atributos que necesitan las preguntas de negocio, cuántos registros utilizables quedan y qué haría falta conseguir.

Pendiente. Depende de las preguntas de negocio del enfoque analítico (actividad 4).

<a id="conclusiones"></a>

## 8. Conclusiones e insights

Los hallazgos de las etapas anteriores, escritos como insights: un entendimiento profundo, no obvio, significativo (con valor potencial), accionable (abre nuevas preguntas) y basado en los datos. Alimentan la sección 6 del reporte y las conclusiones iniciales (actividad 7).

Pendiente.